# H6 — Fase 2: Guided NMF na sentença

Run STM canônico e pinado: `youtube_sent_20260921_190124`. Os quatro
exports ausentes foram recuperados do run de conteúdo idêntico
`youtube_sent_20260920_225112` após comparação SHA-256 do input, θ, β e
final JSON. A comparação com o STM é por posição, após `docs_removed`.

In [ ]:
from pathlib import Path
import sys, json, hashlib, re
import numpy as np
import pandas as pd
from gensim.corpora import Dictionary

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / '03-topic-modeling/configs/params.yaml').exists()), None)
assert PROJECT is not None, 'Execute o notebook dentro do repositorio'
TOPIC_ROOT = PROJECT / '03-topic-modeling'
sys.path.insert(0, str(TOPIC_ROOT / 'notebooks'))
from _helpers import load_params, make_run_output_dir
from _guided import (load_seeds, normalize_seeds, apply_seed_exclusion_rule, build_seed_matrix,
                     category_dependence, bootstrap_prevalence_by_video,
                     coefficient_of_variation, seeded_topic_stability,
                     _run_keyatm_r, guided_nmf, escolher_lambda, scaled_lambda_grid,
                     seed_topic_assignment, topic_concordance, purity_by_group)

params = load_params(str(TOPIC_ROOT / 'configs/params.yaml'))
guided_cfg = params['guided']
seed_path = TOPIC_ROOT / 'configs' / guided_cfg['seeds_file']
seed_sha256 = hashlib.sha256(seed_path.read_bytes()).hexdigest()
seeds = load_seeds(seed_path)

sent_cfg = params['corpora']['youtube_sent']
seeds = normalize_seeds(seeds, extra_stopwords=sent_cfg.get('stopwords_emojis'),
                        preserve_terms=guided_cfg['gnmf'].get('preserve_seed_terms'))
stm_run = guided_cfg['gnmf']['stm_run']
assert stm_run == params.get('nmf_restrito', {}).get('youtube_sent', {}).get('stm_run', stm_run)
stm_dir = TOPIC_ROOT / 'data/output/youtube_sent/stm' / stm_run
required = ['stm_input.csv', 'stm_theta.csv', 'stm_final.json', 'stm_results.csv',
            'stm_metrics.csv', 'stm_prevalence_effects.csv', 'stm_topics_for_eval.csv']
assert all((stm_dir / name).is_file() for name in required), f'R3 incompleto: {stm_dir}'
print('Run STM de sentenca pinado:', stm_run)

## Alinhar as linhas do STM e aplicar a regra de sementes

In [ ]:
stm_input = pd.read_csv(stm_dir / 'stm_input.csv')
stm_theta = pd.read_csv(stm_dir / 'stm_theta.csv').to_numpy()
stm_result = pd.read_csv(stm_dir / 'stm_results.csv')
stm_meta = json.loads((stm_dir / 'stm_final.json').read_text(encoding='utf-8'))
removed = set(stm_meta.get('docs_removed', []))
kept_indices = [index for index in range(len(stm_input)) if index not in removed]
aligned = stm_input.iloc[kept_indices].reset_index(drop=True)
assert len(aligned) == len(stm_theta) == len(stm_result)
assert np.array_equal(stm_theta.argmax(axis=1), stm_result['topic_id'].to_numpy())

tokens = aligned['text'].fillna('').str.split().tolist()
dictionary = Dictionary(tokens)
dictionary.filter_extremes(no_below=sent_cfg['stm_no_below'],
                           no_above=sent_cfg['stm_no_above'])
effective, exclusions = apply_seed_exclusion_rule(
    seeds, dictionary, guided_cfg['min_df_semente']['youtube_sent'],
    guided_cfg['min_sementes_por_aspecto'])
assert effective, 'Nenhum aspecto sobreviveu a regra de exclusao'
Y, aspect_names = build_seed_matrix(effective, dictionary)
print('Sentencas alinhadas:', len(aligned), 'vocabulario:', len(dictionary),
      'aspectos:', aspect_names)

## Matriz TF-IDF esparsa e escolha de λ sem olhar as medidas de H6

In [ ]:
from scipy import sparse
from sklearn.feature_extraction.text import TfidfTransformer

rows, columns, values = [], [], []
for row, doc in enumerate(tokens):
    for column, count in dictionary.doc2bow(doc):
        rows.append(row); columns.append(column); values.append(count)
counts = sparse.csr_matrix((values, (rows, columns)),
                           shape=(len(tokens), len(dictionary)), dtype=float)
X = TfidfTransformer().fit_transform(counts).T.tocsr()
assert sparse.issparse(X) and X.shape == (len(dictionary), len(aligned))
main_free = guided_cfg['gnmf']['livres']
main_seed = guided_cfg['seed_runs'][0]
alpha_grid = guided_cfg['gnmf']['alpha_grid']
lambda_grid = scaled_lambda_grid(X, Y, alpha_grid)
lambda_diagnostics = []
lam = escolher_lambda(X, Y, k=len(aspect_names) + main_free, seed=main_seed,
                     grid=lambda_grid, max_iter=500, diagnostics=lambda_diagnostics,
                     top_n=guided_cfg['gnmf']['regra_lambda_top_n'],
                     frac_min=guided_cfg['gnmf']['regra_lambda_frac'])
lambda_rule_met = any(row['min_seed_recall'] >= guided_cfg['gnmf']['regra_lambda_frac']
                      for row in lambda_diagnostics)
print('Lambda fixado antes da concordancia STM:', lam, 'regra:', lambda_rule_met)

## Ajustar os modelos e exportar as medidas

In [ ]:
run_dir = make_run_output_dir(TOPIC_ROOT / 'data/output/youtube_sent/guided', 'gnmf')
(run_dir / 'seeds_efetivas.json').write_text(
    json.dumps({'aspectos': effective, **exclusions}, ensure_ascii=False, indent=2), encoding='utf-8')
stm_dominant = stm_theta.argmax(axis=1)
aspect_topics = guided_cfg['gnmf']['aspect_topics']
assert aspect_topics == params.get('nmf_restrito', {}).get('youtube_sent', {}).get('aspect_topics', aspect_topics)
video_ids = aligned['post_id'].astype(str).tolist()
categories = aligned['product_category'].fillna('outras').astype(str).tolist()
stm_cv = category_dependence(stm_theta, categories, video_ids).drop_duplicates('topico').set_index('topico')['cv']
all_runs = {}
free_values = [main_free, *guided_cfg['gnmf']['livres_sens']] if lambda_rule_met else [main_free]
for n_free in free_values:
    for seed in guided_cfg['seed_runs']:
        subdir = run_dir / f'livres_{n_free}_seed_{seed}'
        subdir.mkdir(parents=True, exist_ok=True)
        result = guided_nmf(X, Y, k=len(aspect_names) + n_free,
                            lam=lam, seed=seed, max_iter=500)
        A, S, B = result['A'], result['S'], result['B']
        assignment = seed_topic_assignment(B)
        seed_recall = [len(set(np.argsort(-A[:, assignment[j]])[:guided_cfg['gnmf']['regra_lambda_top_n']])
                           & set(np.flatnonzero(Y[:, j]))) / int(np.count_nonzero(Y[:, j]))
                       for j in range(len(aspect_names))]
        dominant = S.argmax(axis=0)
        normalized = S / np.maximum(S.sum(axis=0, keepdims=True), 1e-12)
        pd.DataFrame({'topico': dominant, 'peso': normalized.max(axis=0)}).to_csv(
            subdir / 'w_sentenca.csv', index=False)
        topic_rows = []
        for topic in range(A.shape[1]):
            aspect = next((aspect_names[j] for j, t in enumerate(assignment) if t == topic), None)
            for rank, word_id in enumerate(np.argsort(-A[:, topic])[:20], start=1):
                word = dictionary[int(word_id)]
                topic_rows.append({'topico': topic, 'aspecto': aspect, 'rank': rank,
                                   'palavra': word, 'peso': A[word_id, topic],
                                   'semente': bool(aspect and word in effective[aspect])})
        pd.DataFrame(topic_rows).to_csv(subdir / 'topics.csv', index=False)
        concordance = topic_concordance(stm_dominant, dominant)
        select = np.isin(stm_dominant, aspect_topics)
        purity = purity_by_group(dominant[select], stm_dominant[select],
                                 allowed_topics=assignment)
        purity.to_csv(subdir / 'concordancia_stm.csv', index=False)
        pd.crosstab(stm_dominant[select], dominant[select],
                    rownames=['stm'], colnames=['guided']).to_csv(subdir / 'contingencia_stm.csv')
        by_category = category_dependence(
            normalized.T, categories, video_ids,
            n_boot=guided_cfg['gnmf']['bootstrap_runs'], seed=seed)
        by_category.to_csv(subdir / 'prevalencia_categoria.csv', index=False)
        from scipy.stats import spearmanr
        guided_cv = by_category.drop_duplicates('topico').set_index('topico')['cv']
        comparable = (purity[purity['topico_majoritario'].isin(assignment)]
                      .sort_values('fracao_majoritaria', ascending=False)
                      .drop_duplicates('topico_majoritario'))
        cv_pairs = [(float(stm_cv.loc[int(row.grupo)]),
                     float(guided_cv.loc[int(row.topico_majoritario)]))
                    for row in comparable.itertuples()]
        rho = float(spearmanr(*zip(*cv_pairs)).statistic) if len(cv_pairs) >= 2 else float('nan')
        pd.DataFrame([{'spearman_cv': rho, 'n_aspectos_comuns': len(cv_pairs)}]).to_csv(
            subdir / 'spearman_cv_stm.csv', index=False)
        all_runs[(n_free, seed)] = {'dir': str(subdir), 'nmi': concordance['nmi'],
                                   'ari': concordance['ari'],
                                   'pureza_ge_05': int((purity.fracao_majoritaria >= 0.5).sum()),
                                   'min_seed_recall': min(seed_recall),
                                   'loss_final': result['loss_history'][-1]}
        del result, A, S, B

metadata = {'stm_run': stm_run, 'protocol': guided_cfg['gnmf']['protocol'],
            'lambda': lam, 'alpha_grid': alpha_grid, 'lambda_grid': lambda_grid,
            'lambda_diagnostics': lambda_diagnostics,
            'preserve_seed_terms': guided_cfg['gnmf']['preserve_seed_terms'],
            'seed_file_sha256': seed_sha256,
            'seeds_commit': None, 'effective_seeds': effective,
            'exclusions': exclusions,
            'lambda_rule_met': lambda_rule_met,
            'runs': {str(key): value for key, value in all_runs.items()}}
(run_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saida:', run_dir)

## Interpretação pré-registrada

Concordância sustentada: pureza ≥0,5 em pelo menos seis dos oito temas
de aspecto do STM. Reportar NMI/ARI, CV, razão laptop/phone, IC por vídeo,
correlação de Spearman dos CVs com a Tabela §5.2 e estabilidade Jaccard
entre seeds. Registrar o resultado em `docs/idealizacao-artigo.txt` §5.

In [ ]:
word_runs = []
for seed in guided_cfg['seed_runs']:
    topics = pd.read_csv(Path(all_runs[(main_free, seed)]['dir']) / 'topics.csv')
    word_runs.append({aspect: topics.loc[topics.aspecto == aspect, 'palavra'].head(10).tolist()
                      for aspect in aspect_names})
seeded_topic_stability(word_runs).to_csv(run_dir / 'estabilidade_jaccard.csv', index=False)
print('Resumo:', all_runs[(main_free, main_seed)])
print('Registre a interpretacao humana e a correlacao de Spearman em docs/idealizacao-artigo.txt §5.')